# LINEBENCH — 02 Heuristics Comparison & Portfolio Analysis

This notebook demonstrates and analyzes the heuristic balancing methods:
1. **Worked Example:** Visualizes the Bowman-8 precedence graph and step-by-step balancing.
2. **Priority Rules Comparison:** Evaluates the 6 rules (`RPW`, `UBT`, `MTS`, `MDS`, `EST`, `LST`) in both directions (Forward vs. Backward).
3. **Randomized Multi-Pass:** Analyzes convergence as number of random passes $K$ increases.
4. **Portfolio Performance:** Determines which heuristic wins where and evaluates overall portfolio effectiveness.

In [ ]:
from pathlib import Path
import networkx as nx
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

from linebench.instances.loader import load_in2
from linebench.instances.measures import compute_instance_measures
from linebench.heuristics.construct import solve_station_oriented
from linebench.heuristics.priorities import PRIORITY_RULE_REGISTRY
from linebench.heuristics.random_pass import solve_randomized_multi_pass
from linebench.heuristics.portfolio import evaluate_heuristic_portfolio

sns.set_theme(style="whitegrid")

# Load Bowman-8 instance (c=20, published optimum m*=5)
instance = load_in2("../data/raw/scholl1993/precedence graphs/BOWMAN8.IN2", cycle_time=20)
measures = compute_instance_measures(instance)
print(f"Instance: {instance.name}, Tasks: {instance.num_tasks}, Cycle Time: {instance.cycle_time}, Lower Bound: {measures.lb}")

In [ ]:
# 1. Diagram of Bowman-8 Precedence Graph
G = nx.DiGraph()
for t_id, d in instance.task_times.items():
    G.add_node(t_id, label=f"Task {t_id}\n(t={d})")
G.add_edges_from(instance.precedences)

plt.figure(figsize=(9, 5))
pos = nx.spring_layout(G, seed=42)
labels = nx.get_node_attributes(G, "label")
nx.draw(G, pos, with_labels=True, labels=labels, node_color="lightsteelblue",
        node_size=2000, font_size=9, font_weight="bold", arrows=True, arrowsize=18)
plt.title("Bowman-8 Assembly Line Precedence Graph (n=8, c=20)")
plt.show()

In [ ]:
# 2. Compare Deterministic Priority Rules (Forward vs Backward)
rule_records = []
for rule in PRIORITY_RULE_REGISTRY:
    for direction in ["forward", "backward"]:
        sol = solve_station_oriented(instance, priority_rule=rule, direction=direction)
        rule_records.append({
            "rule": rule,
            "direction": direction,
            "stations": sol.num_stations,
            "runtime_ms": sol.duration_seconds * 1000,
            "feasible": sol.is_feasible,
            "loads": list(sol.station_loads.values())
        })

df_rules = pd.DataFrame(rule_records)
df_rules

In [ ]:
# Bar plot comparing stations used by heuristic rule and direction
plt.figure(figsize=(10, 4.5))
ax = sns.barplot(data=df_rules, x="rule", y="stations", hue="direction", palette="muted")
plt.axhline(5, color="crimson", linestyle="--", label="Known Optimum m*=5")
plt.title("Bowman-8: Stations Found by Heuristic Rule and Direction")
plt.ylabel("Number of Stations (m)")
plt.xlabel("Priority Rule")
plt.legend()
plt.ylim(0, 8)
plt.show()

In [ ]:
# 3. Randomized Multi-Pass Convergence (Passes K sweep)
k_values = [1, 5, 10, 25, 50, 100, 250, 500]
sweep_records = []
for k in k_values:
    mp = solve_randomized_multi_pass(instance, num_passes=k, seed=1001, priority_rule="RPW")
    sweep_records.append({
        "passes_k": k,
        "best_stations": mp.best_solution.num_stations,
        "distinct_optimal_balances": len(mp.distinct_best_balances),
        "runtime_ms": mp.duration_seconds * 1000
    })

df_sweep = pd.DataFrame(sweep_records)
df_sweep

In [ ]:
# 4. Heuristic Portfolio Evaluation
portfolio = evaluate_heuristic_portfolio(instance, multi_pass_passes=100)
print(f"Portfolio Best Stations: {portfolio.best_stations} (Optimum m*=5)")
print(f"Winning Methods: {portfolio.winning_methods}")
print(f"Total Heuristic Runtime: {portfolio.total_runtime_seconds * 1000:.2f} ms")
print(f"All Solutions Feasible: {portfolio.all_feasible}")